In [1]:
# 1
# Import libraries
# 必要なライブラリを読み込む

import re
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd

import geopandas as gpd

In [2]:
# 2
# Define data paths
# データファイルのパスを設定する

data_dir = Path(".")

iom_path = data_dir / "Baseline April 2026_public.xlsx"

hot_path = data_dir / "populated_places.gpkg"

In [3]:
# 3
# Read and inspect source datasets
# 元データを読み込み、基本構造を確認する

iom = pd.read_excel(iom_path)

hot = gpd.read_file(hot_path)

print("\nSource dataset summary")

print(
    f"IOM: {iom.shape[0]:,} rows × "
    f"{iom.shape[1]:,} columns"
)

print(
    f"HOT: {hot.shape[0]:,} rows × "
    f"{hot.shape[1]:,} columns"
)

print("\nHOT geometry types:")

print(hot.geometry.geom_type.value_counts())

print("\nHOT CRS:", hot.crs)


Source dataset summary
IOM: 8,394 rows × 20 columns
HOT: 29,863 rows × 21 columns

HOT geometry types:
Polygon         20840
Point            8995
MultiPolygon       27
LineString          1
Name: count, dtype: int64

HOT CRS: EPSG:4326


In [4]:
# 4
# Clean IOM column names
# IOMの列名を整形する

iom.columns = (
    iom.columns
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [5]:
# 5
# Inspect cleaned IOM column names
# pandasで読み込み、整形したIOM列名を確認する
# rename前に、DataFrame上の正確な列名を確認する

for i, column in enumerate(iom.columns, start=1):
    print(f"{i:2d}. {repr(column)}")

 1. 'Governorate Pcode'
 2. 'Governorate'
 3. 'District Pcode'
 4. 'District'
 5. 'Sub-District Pcode'
 6. 'Sub-district'
 7. 'admin 4 Pcode'
 8. 'admin4'
 9. 'location Type'
10. 'Location Name'
11. 'Location Pcode'
12. 'Residents'
13. 'IDP Returnees from December 2024 until now'
14. 'IDP Returnees only in 2026'
15. 'IDPs'
16. 'Arrivals from abroad to place of originfrom December 2024 until now'
17. 'Arrivals from abroad to place of origin only in 2026'
18. 'Arrivals from abroad to another place from December 2024 until now'
19. 'Arrivals from abroad to another place only in 2026'
20. 'Total Population'


In [6]:
# 6
# Rename IOM columns for analysis
# IOMの列名を分析用に変更する

iom = iom.rename(columns={

    "Sub-District Pcode": "adm3_pcode",
    "Sub-district": "adm3_name",

    "Location Pcode": "location_pcode",
    "Location Name": "location_name",

    "Residents": "residents",
    "IDPs": "idps",
    "Total Population": "total_population",

    "IDP Returnees from December 2024 until now":
        "idp_returnees_dec2024",

    "IDP Returnees only in 2026":
        "idp_returnees_2026"

})

In [7]:
# 7
# Validate required IOM columns
# IOMの必須列を検証する

required_iom_columns = [
    "adm3_pcode",
    "adm3_name",
    "location_pcode",
    "location_name",
    "residents",
    "idps",
    "idp_returnees_dec2024",
    "idp_returnees_2026",
    "total_population"
]

missing_iom_columns = [
    column
    for column in required_iom_columns
    if column not in iom.columns
]

if missing_iom_columns:
    raise KeyError(
        f"Missing required IOM columns: "
        f"{missing_iom_columns}"
    )

print("Required IOM columns validated successfully.")

Required IOM columns validated successfully.


In [8]:
# 8
# Validate required HOT columns
# HOTの必須列を検証する

required_hot_columns = [
    "adm3_pcode",
    "name_en",
    "geometry"
]

missing_hot_columns = [
    column
    for column in required_hot_columns
    if column not in hot.columns
]

if missing_hot_columns:
    raise KeyError(
        f"Missing required HOT columns: "
        f"{missing_hot_columns}"
    )

print("Required HOT columns validated successfully.")

Required HOT columns validated successfully.


In [9]:
# 9
# Create analysis-ready working datasets
# 分析用の作業データセットを作成する

# IOM: retain settlements with the identifiers
# and names required for matching

iom_work = iom[
    iom["adm3_pcode"].notna()
    & iom["location_pcode"].notna()
    & iom["location_name"].notna()
].copy()

# HOT: retain named settlement points
# with an ADM3 Pcode

hot_work = hot[
    hot.geometry.notna()
    & (hot.geometry.geom_type == "Point")
    & hot["name_en"].notna()
    & hot["adm3_pcode"].notna()
].copy()

In [10]:
# 10
# Inspect working dataset sizes
# 分析対象データの件数を確認する

print(
    f"IOM working dataset: {len(iom_work):,} rows "
    f"({len(iom) - len(iom_work):,} excluded)"
)

print(
    f"HOT working dataset: {len(hot_work):,} rows "
    f"({len(hot) - len(hot_work):,} excluded)"
)

IOM working dataset: 8,394 rows (0 excluded)
HOT working dataset: 7,969 rows (21,894 excluded)


In [11]:
# 11
# Preview selected IOM working columns
# 分析対象となるIOMの主要列を確認する

iom_work[
    [
    "adm3_pcode",
    "adm3_name",
    "location_pcode",
    "location_name",
    "residents",
    "idps",
    "idp_returnees_dec2024",
    "idp_returnees_2026",
    "total_population"
]
].head()

,adm3_pcode,adm3_name,location_pcode,location_name,residents,idps,idp_returnees_dec2024,idp_returnees_2026,total_population
0,SY060400,Al-Qardaha,C3808,Ein Elarus,1165,0,0,0,1172
1,SY060001,Bahlolieh,C3494,Amruniyeh,1380,25,0,0,1405
2,SY060300,Al-Haffa,C3677,Hara,1984,0,0,0,1984
3,SY060300,Al-Haffa,C3670,Samiya,1915,0,0,0,1915
4,SY060204,Dalyeh,C3648,Ein Ghannam,1097,0,0,0,1097


In [12]:
# 12
# Preview selected HOT working columns
# 分析対象となるHOTの主要列を確認する

hot_preview_columns = [
    column
    for column in [
        "adm3_pcode",
        "adm3_name",
        "name_en",
        "place",
        "population",
        "geometry"
    ]
    if column in hot_work.columns
]

hot_work[hot_preview_columns].head()

,adm3_pcode,adm3_name,name_en,place,population,geometry
2,SY010000,Damascus,Damascus,city,1414913,POINT (36.30958 33.51307)
24,SY010000,Damascus,Dwelaa,town,None,POINT (36.32742 33.50157)
50,SY030103,Jaramana,Ain Terma,village,35722,POINT (36.34892 33.51441)
67,SY030104,Maliha,Jisrein,village,9442,POINT (36.38843 33.51039)
69,SY030105,Kafr Batna,Kafr Batna,village,22535,POINT (36.37358 33.5126)


In [13]:
# 13
# Normalise settlement names
# 集落名を照合用に正規化する


# Characters treated as separators
# ハイフン・アポストロフィを空白として扱う
HYPHENS_AND_APOSTROPHES = "-‐-‒–—'’‘"


def normalise_name(name):
    """
    Normalise settlement names for comparison.

    - Convert missing values to an empty string
    - Convert text to lowercase
    - Replace hyphens and apostrophes with spaces
    - Collapse repeated whitespace
    """

    if pd.isna(name):
        return ""

    name = str(name).lower()

    name = re.sub(
        f"[{re.escape(HYPHENS_AND_APOSTROPHES)}]",
        " ",
        name
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    )

    return name.strip()


# Apply normalisation
# IOMとHOTの集落名に正規化処理を適用する

iom_work["name_normalised"] = (
    iom_work["location_name"]
    .apply(normalise_name)
)

hot_work["name_normalised"] = (
    hot_work["name_en"]
    .apply(normalise_name)
)

In [14]:
# 13.5 check

test_names = [
    "Al-Qardaha",
    "Ain Terma",
    "Kafr-Batna",
    "Ma'arrat An Nu'man",
    "  Abu   Jarada  "
]

for name in test_names:
    print(f"{name!r} -> {normalise_name(name)!r}")

'Al-Qardaha' -> 'al qardaha'
'Ain Terma' -> 'ain terma'
'Kafr-Batna' -> 'kafr batna'
"Ma'arrat An Nu'man" -> 'ma arrat an nu man'
'  Abu   Jarada  ' -> 'abu jarada'


In [15]:
# 14
# Preview normalised settlement names
# 正規化された地名を確認する

print("IOM")

display(
    iom_work[
        ["location_name", "name_normalised"]
    ].head(10)
)

print("\nHOT")

display(
    hot_work[
        ["name_en", "name_normalised"]
    ].head(10)
)

IOM


,location_name,name_normalised
0,Ein Elarus,ein elarus
1,Amruniyeh,amruniyeh
2,Hara,hara
3,Samiya,samiya
4,Ein Ghannam,ein ghannam
5,Kherbet Hishun,kherbet hishun
6,Doha,doha
7,Mokhtariyeh,mokhtariyeh
8,Herf Elsari,herf elsari
9,Sakhaba,sakhaba



HOT


,name_en,name_normalised
2,Damascus,damascus
24,Dwelaa,dwelaa
50,Ain Terma,ain terma
67,Jisrein,jisrein
69,Kafr Batna,kafr batna
75,Hezze,hezze
99,Saqba,saqba
108,Hamura,hamura
213,Zamalka,zamalka
232,Jobar,jobar


In [16]:
# 15
# Calculate settlement name similarity
# 地名の類似度を計算する

def similarity_score(name_a, name_b):
    """
    2つの正規化済み地名を比較し、
    0〜100の類似度を返す。
    """

    # どちらかが空欄なら比較しない
    if not name_a or not name_b:
        return 0.0

    score = SequenceMatcher(
        None,
        str(name_a),
        str(name_b)
    ).ratio()

    return score * 100

In [17]:
# 16
# Test the settlement name similarity function
# 地名の類似度計算を確認する

print(
    "完全一致:",
    round(similarity_score("abu jarada", "abu jarada"), 1)
)

print(
    "表記揺れ:",
    round(similarity_score("thahr el arab", "dahr al arab"), 1)
)

print(
    "別の地名:",
    round(similarity_score("damascus", "sallurin"), 1)
)

完全一致: 100.0
表記揺れ: 80.0
別の地名: 25.0


In [18]:
# 17
# Build lookup table across all common ADM3 areas
# すべての共通ADM3を対象にLookup Tableを自動作成する

# 照合結果を格納する空のリスト
lookup_rows = []

# IOMとHOTの両方に存在するADM3 PCodeを取得
common_adm3_pcodes = sorted(
    set(iom_work["adm3_pcode"].dropna())
    & set(hot_work["adm3_pcode"].dropna())
)

print("共通するADM3 PCode数:", len(common_adm3_pcodes))


# 共通するADM3を1つずつ処理
for adm3_pcode in common_adm3_pcodes:

    # 現在のADM3に属するIOM Locationを抽出
    iom_group = iom_work[
        iom_work["adm3_pcode"] == adm3_pcode
    ].copy()

    # 現在のADM3に属するHOT Pointを抽出
    hot_group = hot_work[
        hot_work["adm3_pcode"] == adm3_pcode
    ].copy()

    # IOM Locationを1件ずつ処理
    for iom_index, iom_row in iom_group.iterrows():

        iom_name = iom_row["name_normalised"]

        # IOM地名が空欄の場合
        if not iom_name:
            lookup_rows.append({
                "adm3_pcode": adm3_pcode,
                "iom_index": iom_index,
                "iom_location_pcode": iom_row["location_pcode"],
                "iom_name": iom_row["location_name"],
                "hot_index": None,
                "hot_name_candidate": None,
                "hot_place": None,
                "hot_population": None,
                "similarity_score": 0.0,
                "match_status": "Missing IOM name"
            })

            continue

        # 同じADM3内のHOT候補を比較用にコピー
        candidates = hot_group.copy()

        # IOM地名と各HOT地名の類似度を計算
        candidates["similarity_score"] = (
            candidates["name_normalised"]
            .apply(
                lambda hot_name: similarity_score(
                    iom_name,
                    hot_name
                )
            )
        )

        # 類似度が最も高いHOT候補を1件取得
        best_match_index = candidates["similarity_score"].idxmax()
        best_match = candidates.loc[best_match_index]

        best_score_raw = best_match["similarity_score"]
        best_score = round(best_score_raw, 1)

        # スコアに応じて確認状態を付与
        if iom_name == best_match["name_normalised"]:
            match_status = "Exact"
        elif best_score_raw >= 85:
            match_status = "High confidence"
        elif best_score_raw >= 70:
            match_status = "Review"
        else:
            match_status = "Low confidence"

        # 照合結果をリストへ追加
        lookup_rows.append({
            "adm3_pcode": adm3_pcode,
            "iom_index": iom_index,
            "iom_location_pcode": iom_row["location_pcode"],
            "iom_name": iom_row["location_name"],
            "hot_index": best_match.name,
            "hot_name_candidate": best_match["name_en"],
            "hot_place": best_match.get("place"),
            "hot_population": best_match.get("population"),
            "similarity_score": best_score,
            "match_status": match_status
        })


# 照合結果をDataFrameへ変換
lookup_table_all = pd.DataFrame(lookup_rows)

# ADM3ごとに、類似度の高い順へ並べ替え
lookup_table_all = (
    lookup_table_all
    .sort_values(
        by=["adm3_pcode", "similarity_score"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)


# 処理結果を確認
print("Lookup Table rows:", len(lookup_table_all))
print(
    "Processed ADM3:",
    lookup_table_all["adm3_pcode"].nunique()
)

lookup_table_all.head(20)

共通するADM3 PCode数: 267
Lookup Table rows: 8391
Processed ADM3: 267


,adm3_pcode,iom_index,iom_location_pcode,iom_name,hot_index,hot_name_candidate,hot_place,hot_population,similarity_score,match_status
0,SY010000,54,C1002,Yarmuk,15294,Al-Yarmuk,city,137248,80.0,Review
1,SY010000,8167,N0007,Qaboun (ne),235,Qaboun,town,None,70.6,Review
2,SY010000,8066,N0040,Saroujah (ne),15586,Sarouja,town,None,70.0,Review
3,SY010000,7982,N0057,Amin - Damascus (ne),2,Damascus,city,1414913,61.5,Low confidence
4,SY010000,7961,N0021,Al Madaris (ne),15312,Al Midan,town,None,60.9,Low confidence
5,SY010000,8268,N0028,Al Mazraa (ne),15361,Al Mazzeh,town,None,60.9,Low confidence
6,SY010000,8019,N0015,Nuzha - Damascus (ne),2,Damascus,city,1414913,59.3,Low confidence
7,SY010000,8260,N0071,Jalaa - Damascus (ne),2,Damascus,city,1414913,59.3,Low confidence
8,SY010000,8059,N0079,Rawdet Al Midan (ne),15312,Al Midan,town,None,57.1,Low confidence
9,SY010000,8146,N0094,Al Qadam (ne),15312,Al Midan,town,None,57.1,Low confidence


In [19]:
#18
# Validate lookup table quality
# Lookup Tableの品質を確認する

# Lookup Tableに含まれなかったIOM行を抽出
unmatched_iom_rows = iom_work[
    ~iom_work.index.isin(
        lookup_table_all["iom_index"]
    )
].copy()

print("IOM working rows:", len(iom_work))
print("Lookup Table rows:", len(lookup_table_all))
print("Unmatched IOM rows:", len(unmatched_iom_rows))

display(
    unmatched_iom_rows[
        [
            "adm3_pcode",
            "adm3_name",
            "location_pcode",
            "location_name"
        ]
    ]
)


# 同じHOT Pointが複数のIOM Locationに選ばれた回数を確認
hot_candidate_counts = (
    lookup_table_all["hot_index"]
    .dropna()
    .value_counts()
)

duplicated_hot_indexes = hot_candidate_counts[
    hot_candidate_counts > 1
]

print(
    "HOT Points selected multiple times:",
    len(duplicated_hot_indexes)
)

maximum_selections = (
    int(hot_candidate_counts.max())
    if not hot_candidate_counts.empty
    else 0
)

print(
    "Maximum number of selections for one HOT Point:",
    maximum_selections
)

IOM working rows: 8394
Lookup Table rows: 8391
Unmatched IOM rows: 3


,adm3_pcode,adm3_name,location_pcode,location_name
3545,SY100001,Arwad,C5257,Arwad
7144,SY030902,Hajar Aswad,C2502,Hajar Aswad
7801,SY030303,Raheiba,C2394,Raheiba


HOT Points selected multiple times: 1622
Maximum number of selections for one HOT Point: 33


In [20]:
# 19
# Diagnose unmatched IOM rows
# 未照合IOM行の原因を確認する

unmatched_adm3_pcodes = unmatched_iom_rows[
    "adm3_pcode"
].unique()

print("Unmatched ADM3 PCode:")
print(unmatched_adm3_pcodes)

unmatched_hot_preview_columns = [
    column
    for column in [
        "name_en",
        "place",
        "adm3_pcode",
        "adm3_name",
        "geometry"
    ]
    if column in hot.columns
]

display(
    hot[
        hot["adm3_pcode"].isin(unmatched_adm3_pcodes)
    ][unmatched_hot_preview_columns]
)

Unmatched ADM3 PCode:
['SY100001' 'SY030902' 'SY030303']


,name_en,place,adm3_pcode,adm3_name,geometry
1866,None,None,SY030303,Raheiba,"POLYGON ((37.09568 33.87483, 37.09576 33.87491..."
1867,None,None,SY030303,Raheiba,"POLYGON ((36.98076 33.74924, 36.98329 33.74987..."
1880,None,None,SY030303,Raheiba,"POLYGON ((36.68775 33.74777, 36.68771 33.74912..."


In [21]:
# 20
# Inspect duplicated HOT candidates
# 複数回選ばれたHOT候補を確認する

if hot_candidate_counts.empty:
    print("No HOT candidates were selected.")

else:
    most_selected_hot_index = hot_candidate_counts.idxmax()

    print("Most selected HOT index:", most_selected_hot_index)
    print(
        "Number of selections:",
        hot_candidate_counts.loc[most_selected_hot_index]
    )

    display(
        lookup_table_all[
            lookup_table_all["hot_index"] == most_selected_hot_index
        ][
            [
                "adm3_pcode",
                "iom_location_pcode",
                "iom_name",
                "hot_name_candidate",
                "similarity_score",
                "match_status"
            ]
        ].sort_values(
            by="similarity_score",
            ascending=False
        )
    )

Most selected HOT index: 15312
Number of selections: 33


,adm3_pcode,iom_location_pcode,iom_name,hot_name_candidate,similarity_score,match_status
4,SY010000,N0021,Al Madaris (ne),Al Midan,60.9,Low confidence
8,SY010000,N0079,Rawdet Al Midan (ne),Al Midan,57.1,Low confidence
9,SY010000,N0094,Al Qadam (ne),Al Midan,57.1,Low confidence
15,SY010000,N0011,Al Masani (ne),Al Midan,54.5,Low confidence
16,SY010000,N0036,Al Maliki (ne),Al Midan,54.5,Low confidence
17,SY010000,N0072,Al Ikhlas (ne),Al Midan,54.5,Low confidence
18,SY010000,N0003,Al Manara (ne),Al Midan,54.5,Low confidence
21,SY010000,N0095,Al Mustafa (ne),Al Midan,52.2,Low confidence
23,SY010000,N0027,Al Marabit (ne),Al Midan,52.2,Low confidence
27,SY010000,N0020,Al Arin (ne),Al Midan,50.0,Low confidence


In [22]:
# 21
# Inspect match status distribution
# 類似度区分ごとの件数を確認する

match_status_summary = (
    lookup_table_all["match_status"]
    .value_counts()
    .rename_axis("match_status")
    .reset_index(name="rows")
)

match_status_summary["percentage"] = (
    match_status_summary["rows"]
    / len(lookup_table_all)
    * 100
).round(1)

display(match_status_summary)

,match_status,rows,percentage
0,Low confidence,3737,44.5
1,Review,2211,26.3
2,Exact,1233,14.7
3,High confidence,1210,14.4


In [23]:
# 22
# Summarise data matching workflow
# データ照合処理の概要を確認する

print("PROJECT 10 SUMMARY")

print("Original IOM rows :", len(iom))
print("Working IOM rows  :", len(iom_work))

print("Original HOT rows :", len(hot))
print("Working HOT rows  :", len(hot_work))

print("Lookup rows       :", len(lookup_table_all))
print("Unmatched IOM rows:", len(unmatched_iom_rows))

print(
    "Exact rows        :",
    (lookup_table_all["match_status"] == "Exact").sum()
)

print(
    "High confidence   :",
    (lookup_table_all["match_status"] == "High confidence").sum()
)

print(
    "Review rows       :",
    (lookup_table_all["match_status"] == "Review").sum()
)

print(
    "Low confidence    :",
    (lookup_table_all["match_status"] == "Low confidence").sum()
)

print(
    "Missing IOM name  :",
    (lookup_table_all["match_status"] == "Missing IOM name").sum()
)

print(
    "Processed ADM3    :",
    lookup_table_all["adm3_pcode"].nunique()
)

PROJECT 10 SUMMARY
Original IOM rows : 8394
Working IOM rows  : 8394
Original HOT rows : 29863
Working HOT rows  : 7969
Lookup rows       : 8391
Unmatched IOM rows: 3
Exact rows        : 1233
High confidence   : 1210
Review rows       : 2211
Low confidence    : 3737
Missing IOM name  : 0
Processed ADM3    : 267


In [24]:
# 23
# Attach HOT geometry to the lookup table
# HOTのgeometryをLookup Tableへ追加する

# HOT側からgeometryのみを準備する
hot_geometry = hot_work[
    ["geometry"]
].copy()

# HOTの元indexを結合キーとして列へ追加
hot_geometry["hot_index"] = hot_geometry.index


# Lookup TableとHOT geometryを結合
lookup_geo = lookup_table_all.merge(
    hot_geometry,
    on="hot_index",
    how="left",
    validate="many_to_one"
)


# GeoDataFrameへ変換
lookup_geo = gpd.GeoDataFrame(
    lookup_geo,
    geometry="geometry",
    crs=hot_work.crs
)


# 結果を確認
print("Lookup Table rows:", len(lookup_table_all))
print("GeoDataFrame rows:", len(lookup_geo))
print("Geometry missing:", lookup_geo.geometry.isna().sum())

print("Geometry type:")
print(
    lookup_geo.geometry
    .geom_type
    .value_counts(dropna=False)
)

print("CRS:", lookup_geo.crs)

lookup_geo.head()

Lookup Table rows: 8391
GeoDataFrame rows: 8391
Geometry missing: 0
Geometry type:
Point    8391
Name: count, dtype: int64
CRS: EPSG:4326


,adm3_pcode,iom_index,iom_location_pcode,iom_name,hot_index,hot_name_candidate,hot_place,hot_population,similarity_score,match_status,geometry
0,SY010000,54,C1002,Yarmuk,15294,Al-Yarmuk,city,137248,80.0,Review,POINT (36.3048 33.47248)
1,SY010000,8167,N0007,Qaboun (ne),235,Qaboun,town,None,70.6,Review,POINT (36.33427 33.54386)
2,SY010000,8066,N0040,Saroujah (ne),15586,Sarouja,town,None,70.0,Review,POINT (36.29827 33.51731)
3,SY010000,7982,N0057,Amin - Damascus (ne),2,Damascus,city,1414913,61.5,Low confidence,POINT (36.30958 33.51307)
4,SY010000,7961,N0021,Al Madaris (ne),15312,Al Midan,town,None,60.9,Low confidence,POINT (36.29691 33.49329)


In [25]:
# 24
# Attach IOM humanitarian indicators
# IOMの人道指標をGeoDataFrameへ追加する

# IOM側から地図表示に使用する指標を準備
iom_indicators = iom_work[
    [
        "residents",
        "idps",
        "idp_returnees_dec2024",
        "idp_returnees_2026",
        "total_population"
    ]
].copy()

# IOMの元indexを結合キーとして列へ追加
iom_indicators["iom_index"] = iom_indicators.index


# Lookup GeoDataFrameへIOMの人道指標を結合
lookup_geo = lookup_geo.merge(
    iom_indicators,
    on="iom_index",
    how="left",
    validate="one_to_one"
)


# GeoDataFrameとして再設定
lookup_geo = gpd.GeoDataFrame(
    lookup_geo,
    geometry="geometry",
    crs=hot_work.crs
)


# 結果を確認
print("Rows:", len(lookup_geo))
print(
    "Residents missing:",
    lookup_geo["residents"].isna().sum()
)
print(
    "IDPs missing:",
    lookup_geo["idps"].isna().sum()
)
print(
    "IDP returnees since December 2024 missing:",
    lookup_geo["idp_returnees_dec2024"].isna().sum()
)
print(
    "IDP returnees 2026 missing:",
    lookup_geo["idp_returnees_2026"].isna().sum()
)
print(
    "Total population missing:",
    lookup_geo["total_population"].isna().sum()
)
print("CRS:", lookup_geo.crs)

lookup_geo.head()

Rows: 8391
Residents missing: 0
IDPs missing: 0
IDP returnees since December 2024 missing: 0
IDP returnees 2026 missing: 0
Total population missing: 0
CRS: EPSG:4326


,adm3_pcode,iom_index,iom_location_pcode,iom_name,hot_index,hot_name_candidate,hot_place,hot_population,similarity_score,match_status,geometry,residents,idps,idp_returnees_dec2024,idp_returnees_2026,total_population
0,SY010000,54,C1002,Yarmuk,15294,Al-Yarmuk,city,137248,80.0,Review,POINT (36.3048 33.47248),1730,0,1550,0,3820
1,SY010000,8167,N0007,Qaboun (ne),235,Qaboun,town,None,70.6,Review,POINT (36.33427 33.54386),900,600,1050,0,2851
2,SY010000,8066,N0040,Saroujah (ne),15586,Sarouja,town,None,70.0,Review,POINT (36.29827 33.51731),5101,5712,0,0,10903
3,SY010000,7982,N0057,Amin - Damascus (ne),2,Damascus,city,1414913,61.5,Low confidence,POINT (36.30958 33.51307),2400,0,0,0,2400
4,SY010000,7961,N0021,Al Madaris (ne),15312,Al Midan,town,None,60.9,Low confidence,POINT (36.29691 33.49329),9500,2500,0,0,12050


In [26]:
# 25
# Validate IOM location uniqueness
# IOM Locationの一意性を確認する

unique_iom_pcodes = (
    lookup_geo["iom_location_pcode"]
    .nunique()
)

duplicate_iom_pcodes = (
    lookup_geo["iom_location_pcode"]
    .duplicated()
    .sum()
)

unique_hot_points = (
    lookup_geo["hot_index"]
    .dropna()
    .nunique()
)


# 全体件数を確認
print(
    "Lookup GeoDataFrame rows:",
    f"{len(lookup_geo):,}"
)

print(
    "Unique IOM location Pcodes:",
    f"{unique_iom_pcodes:,}"
)

print(
    "Duplicate IOM location Pcodes:",
    f"{duplicate_iom_pcodes:,}"
)

print(
    "Unique HOT Points used:",
    f"{unique_hot_points:,}"
)


# 重複するIOM Location Pcodeの詳細を確認
duplicated_iom_pcode_rows = lookup_geo[
    lookup_geo["iom_location_pcode"].duplicated(
        keep=False
    )
].copy()

if duplicated_iom_pcode_rows.empty:
    print(
        "No duplicated IOM location Pcodes were found."
    )

else:
    display(
        duplicated_iom_pcode_rows[
            [
                "iom_location_pcode",
                "iom_name",
                "adm3_pcode",
                "match_status",
                "similarity_score",
                "hot_index",
                "geometry"
            ]
        ].sort_values(
            by="iom_location_pcode"
        )
    )

Lookup GeoDataFrame rows: 8,391
Unique IOM location Pcodes: 8,391
Duplicate IOM location Pcodes: 0
Unique HOT Points used: 5,942
No duplicated IOM location Pcodes were found.


In [27]:
# 26
# Inspect reused HOT settlement points
# 複数のIOM地点に共有されたHOT Pointを確認する

hot_point_counts = (
    lookup_geo["hot_index"]
    .dropna()
    .value_counts()
)

print(
    "HOT Points used more than once:",
    f"{(hot_point_counts > 1).sum():,}"
)

print(
    "Maximum IOM locations on one HOT Point:",
    f"{hot_point_counts.max():,}"
)

display(
    hot_point_counts
    .head(10)
    .rename_axis("hot_index")
    .reset_index(name="iom_locations")
)

HOT Points used more than once: 1,622
Maximum IOM locations on one HOT Point: 33


,hot_index,iom_locations
0,15312,33
1,19202,13
2,15407,12
3,15361,11
4,235,11
5,27492,10
6,12696,10
7,24331,9
8,12448,9
9,19172,9


In [28]:
# 27
# Prepare data for mapping
# 地図表示用データを準備する

# 信頼度の高い照合結果だけを地図表示に使用する
map_data = lookup_geo[
    lookup_geo["match_status"].isin(
        ["Exact", "High confidence"]
    )
].copy()


# 地図表示前の確認
map_percentage = (
    len(map_data)
    / len(lookup_geo)
    * 100
)

print("Map rows:", len(map_data))
print(
    "Mapped percentage:",
    round(map_percentage, 1),
    "%"
)
print("Geometry missing:", map_data.geometry.isna().sum())
print("CRS:", map_data.crs)

print("\nMap match status:")
print(
    map_data["match_status"]
    .value_counts()
)

map_data.head()

Map rows: 2443
Mapped percentage: 29.1 %
Geometry missing: 0
CRS: EPSG:4326

Map match status:
match_status
Exact              1233
High confidence    1210
Name: count, dtype: int64


,adm3_pcode,iom_index,iom_location_pcode,iom_name,hot_index,hot_name_candidate,hot_place,hot_population,similarity_score,match_status,geometry,residents,idps,idp_returnees_dec2024,idp_returnees_2026,total_population
101,SY020000,2653,C8129,Kafr Tunah,21025,Kafr Tunah,village,None,100.0,Exact,POINT (37.17901 36.33895),0,0,0,0,0
102,SY020000,4188,C7327,Al Buhayrah,19180,Al Buhayrah,village,None,100.0,Exact,POINT (37.15604 36.09261),200,0,35,0,235
103,SY020000,4347,C1009,Abtin,19197,Abtin,village,3081,100.0,Exact,POINT (37.1245 36.04654),3820,0,95,0,3915
104,SY020000,4441,C6323,Azzan,19192,Azzan,hamlet,493,100.0,Exact,POINT (37.15839 36.0635),534,0,165,0,699
105,SY020000,4472,C1018,Qarras,19209,Qarras,village,3326,100.0,Exact,POINT (37.07995 36.09428),200,0,199,0,399


In [29]:
# 2-1
# Import mapping library
# 地図描画用ライブラリを読み込む

import folium

/Users/marisa/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [30]:
# 2-2
# Create base map
# シリア全土のベースマップを作成する

m = folium.Map(
    location=[34.8, 38.5],
    zoom_start=7,
    tiles=(
        "https://{s}.basemaps.cartocdn.com/"
        "light_nolabels/{z}/{x}/{y}{r}.png"
    ),
    attr=(
        '&copy; <a href="https://www.openstreetmap.org/copyright">'
        'OpenStreetMap</a> contributors '
        '&copy; <a href="https://carto.com/attributions">CARTO</a>'
    )
)

In [31]:
# 2-3
# Add administrative boundaries
# 国境線と県境線を追加する

# 国境線：太さ3
folium.GeoJson('syr_admin0.geojson', name='Country',
               style_function=lambda x: {'color': 'black', 'weight': 3, 'fillOpacity': 0}).add_to(m)

# 地域線：太さ1
folium.GeoJson('syr_admin1.geojson', name='Governorate',
               style_function=lambda x: {'color': 'gray', 'weight': 1, 'fillOpacity': 0}).add_to(m)

In [32]:
# 2-4
# Add neighbouring country labels
# 隣国名を追加する

neighbors = {
    "TÜRKIYE": [37.5, 37.5],
    "IRAQ": [34.5, 42.0],
    "JORDAN": [31.9, 36.5],
    "LEBANON": [34.2, 35.0]
}

neighbor_labels = folium.FeatureGroup(
    name="Neighbouring countries",
    show=True
)

for name, coords in neighbors.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f"""
                <div style="
                    font-size: 14pt;
                    font-weight: bold;
                    color: gray;
                    white-space: nowrap;
                    text-align: center;
                    width: 100px;
                    margin-left: -50px;
                ">
                    {name}
                </div>
            """
        )
    ).add_to(neighbor_labels)

neighbor_labels.add_to(m);

In [33]:
# 2-5
# Add governorate labels
# シリア14県の名称を追加する

governorates = {
    "Aleppo": [36.2, 37.5],
    "Al-Hasakeh": [36.5, 40.7],
    "Ar-Raqqa": [36.0, 39.0],
    "As-Sweida": [32.8, 36.9],
    "Daraa": [32.9, 36.2],
    "Deir-ez-Zor": [35.1, 40.5],
    "Damascus": [33.7, 36.7],
    "Hama": [35.2, 37.0],
    "Homs": [34.5, 38.3],
    "Idleb": [35.8, 36.7],
    "Lattakia": [35.6, 36.1],
    "Quneitra": [33.1, 35.9],
    "Rural Damascus": [33.5, 37.5],
    "Tartous": [34.9, 36.1]
}

governorate_labels = folium.FeatureGroup(
    name="Governorate labels",
    show=True
)

for name, coords in governorates.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f"""
                <div style="
                    font-size: 10pt;
                    color: #4b5563;
                    font-weight: 600;
                    white-space: nowrap;
                    text-align: center;
                    width: 100px;
                    margin-left: -50px;
                ">
                    {name}
                </div>
            """
        )
    ).add_to(governorate_labels)

governorate_labels.add_to(m);

In [34]:
# 2-6
# Add matched settlements
# 高信頼で照合された集落を追加する

# Matched settlements レイヤーを作成
matched_settlements = folium.FeatureGroup(
    name="Matched settlements",
    show=False
)

# map_dataのPointを1件ずつ地図へ追加
for _, row in map_data.iterrows():

    folium.CircleMarker(
        location=[
            row.geometry.y,
            row.geometry.x
        ],
        radius=3,
        color="#2b6cb0",
        weight=1,
        fill=True,
        fill_color="#2b6cb0",
        fill_opacity=0.65,
        tooltip=(
            f"{row['iom_name']} | "
            f"{row['match_status']}"
        )
    ).add_to(matched_settlements)

# Matched settlements レイヤーを地図へ追加
matched_settlements.add_to(m);

In [35]:
# 2-7
# Prepare residents data for mapping
# Residentsデータを地図表示用に準備する

residents_geo = map_data[
    map_data["residents"].notna()
    & (map_data["residents"] > 0)
    & map_data.geometry.notna()
].copy()


# Residentsデータを確認
print(f"Residents locations: {len(residents_geo):,}")
print(
    f"Maximum residents: "
    f"{residents_geo['residents'].max():,.0f}"
)


# 地図表示に必要な列を確認
required_columns = [
    "iom_name",
    "residents",
    "idps",
    "idp_returnees_dec2024",
    "idp_returnees_2026",
    "total_population",
    "match_status",
    "similarity_score",
    "geometry"
]

for column in required_columns:
    print(
        column,
        column in map_data.columns
    )

Residents locations: 2,170
Maximum residents: 485,113
iom_name True
residents True
idps True
idp_returnees_dec2024 True
idp_returnees_2026 True
total_population True
match_status True
similarity_score True
geometry True


In [36]:
# 2-8
# Add proportional residents layer
# 居住人口を比例円で表示する

# 人口値を読みやすい形式へ変換する
def format_count(value):
    """
    Format population values with thousands separators.
    欠損値は No data と表示する。
    """

    if pd.isna(value):
        return "No data"

    return f"{int(round(value)):,}"


# 人口値を円の半径へ変換する
def proportional_radius(
    value,
    max_value,
    min_radius=2.5,
    max_radius=18
):
    """
    Scale population values using square-root scaling.
    平方根を使用して、極端に大きな円になることを防ぐ。
    """

    if (
        pd.isna(value)
        or value <= 0
        or pd.isna(max_value)
        or max_value <= 0
    ):
        return min_radius

    scaled_value = (value / max_value) ** 0.5

    return (
        min_radius
        + scaled_value * (max_radius - min_radius)
    )


# Residentsレイヤーを作成
residents_layer = folium.FeatureGroup(
    name="Residents",
    show=True
)


# Residentsの最大値を取得
max_residents = residents_geo["residents"].max()


# 各地点をResidents数に応じた大きさで地図へ追加
for _, row in residents_geo.iterrows():

    residents_value = row["residents"]

    popup_html = f"""
    <div style="
        font-family: Arial;
        font-size: 13px;
        min-width: 220px;
    ">
        <b>Settlement</b><br>
        {row['iom_name']}
        <br><br>

        <b>Residents:</b>
        {format_count(row['residents'])}<br>

        <b>IDPs:</b>
        {format_count(row['idps'])}<br>

        <b>IDP Returnees since Dec 2024:</b>
        {format_count(row['idp_returnees_dec2024'])}<br>

        <b>IDP Returnees in 2026:</b>
        {format_count(row['idp_returnees_2026'])}<br>

        <b>Total Population:</b>
        {format_count(row['total_population'])}
        <br><br>

        <b>Match status:</b>
        {row['match_status']}<br>

        <b>Similarity score:</b>
        {row['similarity_score']}
    </div>
    """

    folium.CircleMarker(
        location=[
            row.geometry.y,
            row.geometry.x
        ],

        radius=proportional_radius(
            residents_value,
            max_residents
        ),

        color="#2563a6",
        weight=1,

        fill=True,
        fill_color="#3182bd",
        fill_opacity=0.55,

        tooltip=(
            f"{row['iom_name']} | "
            f"Residents: {format_count(residents_value)}"
        ),

        popup=folium.Popup(
            popup_html,
            max_width=320
        )

    ).add_to(residents_layer)


# Residentsレイヤーを地図へ追加
residents_layer.add_to(m)


# 結果を確認
print(
    "Residents locations:",
    f"{len(residents_geo):,}"
)

print(
    "Maximum residents:",
    format_count(max_residents)
)

Residents locations: 2,170
Maximum residents: 485,113


In [37]:
# 2-9
# Prepare IDPs data for mapping
# IDPsデータを地図表示用に準備する

idps_geo = map_data[
    map_data["idps"].notna()
    & (map_data["idps"] > 0)
    & map_data.geometry.notna()
].copy()

print(
    "IDP locations:",
    f"{len(idps_geo):,}"
)

print(
    "Maximum IDPs:",
    format_count(idps_geo["idps"].max())
)

IDP locations: 808
Maximum IDPs: 703,000


In [38]:
# 2-10
# Add proportional IDPs layer
# 国内避難民人口を比例円で表示する

# IDPsレイヤーを作成
idps_layer = folium.FeatureGroup(
    name="IDPs",
    show=False
)


# IDPsの最大値を取得
max_idps = idps_geo["idps"].max()


# 各地点をIDPs数に応じた大きさで地図へ追加
for _, row in idps_geo.iterrows():

    idps_value = row["idps"]

    popup_html = f"""
    <div style="
        font-family: Arial;
        font-size: 13px;
        min-width: 220px;
    ">
        <b>Settlement</b><br>
        {row['iom_name']}
        <br><br>

        <b>Residents:</b>
        {format_count(row['residents'])}<br>

        <b>IDPs:</b>
        {format_count(row['idps'])}<br>

        <b>IDP Returnees since Dec 2024:</b>
        {format_count(row['idp_returnees_dec2024'])}<br>

        <b>IDP Returnees in 2026:</b>
        {format_count(row['idp_returnees_2026'])}<br>

        <b>Total Population:</b>
        {format_count(row['total_population'])}
        <br><br>

        <b>Match status:</b>
        {row['match_status']}<br>

        <b>Similarity score:</b>
        {row['similarity_score']}
    </div>
    """

    folium.CircleMarker(
        location=[
            row.geometry.y,
            row.geometry.x
        ],

        radius=proportional_radius(
            idps_value,
            max_idps
        ),

        color="#b45309",
        weight=1,

        fill=True,
        fill_color="#f59e0b",
        fill_opacity=0.60,

        tooltip=(
            f"{row['iom_name']} | "
            f"IDPs: {format_count(idps_value)}"
        ),

        popup=folium.Popup(
            popup_html,
            max_width=320
        )

    ).add_to(idps_layer)


# IDPsレイヤーを地図へ追加
idps_layer.add_to(m)

print(
    "IDPs layer created:",
    f"{len(idps_geo):,}",
    "locations"
)

IDPs layer created: 808 locations


In [39]:
# 2-11
# Prepare IDP returnees since December 2024 data for mapping
# 2024年12月以降のIDP帰還者データを地図表示用に準備する

idp_returnees_dec2024_geo = map_data[
    map_data["idp_returnees_dec2024"].notna()
    & (map_data["idp_returnees_dec2024"] > 0)
    & map_data.geometry.notna()
].copy()

print(
    "IDP returnee locations since December 2024:",
    f"{len(idp_returnees_dec2024_geo):,}"
)

print(
    "Maximum IDP returnees since December 2024:",
    format_count(
        idp_returnees_dec2024_geo[
            "idp_returnees_dec2024"
        ].max()
    )
)

IDP returnee locations since December 2024: 957
Maximum IDP returnees since December 2024: 39,513


In [40]:
# 2-12
# Add proportional IDP returnees since December 2024 layer
# 2024年12月以降のIDP帰還者人口を比例円で表示する

# IDP Returnees since December 2024 レイヤーを作成
idp_returnees_dec2024_layer = folium.FeatureGroup(
    name="IDP Returnees since Dec 2024",
    show=False
)


# IDP Returnees since December 2024 の最大値を取得
max_idp_returnees_dec2024 = (
    idp_returnees_dec2024_geo[
        "idp_returnees_dec2024"
    ].max()
)


# 各地点をIDP Returnees数に応じた大きさで地図へ追加
for _, row in idp_returnees_dec2024_geo.iterrows():

    idp_returnees_dec2024_value = (
        row["idp_returnees_dec2024"]
    )

    popup_html = f"""
    <div style="
        font-family: Arial;
        font-size: 13px;
        min-width: 220px;
    ">
        <b>Settlement</b><br>
        {row['iom_name']}
        <br><br>

        <b>Residents:</b>
        {format_count(row['residents'])}<br>

        <b>IDPs:</b>
        {format_count(row['idps'])}<br>

        <b>IDP Returnees since Dec 2024:</b>
        {format_count(row['idp_returnees_dec2024'])}<br>

        <b>IDP Returnees in 2026:</b>
        {format_count(row['idp_returnees_2026'])}<br>

        <b>Total Population:</b>
        {format_count(row['total_population'])}
        <br><br>

        <b>Match status:</b>
        {row['match_status']}<br>

        <b>Similarity score:</b>
        {row['similarity_score']}
    </div>
    """

    folium.CircleMarker(
        location=[
            row.geometry.y,
            row.geometry.x
        ],

        radius=proportional_radius(
            idp_returnees_dec2024_value,
            max_idp_returnees_dec2024
        ),

        color="#7A1F3D",
        weight=1,

        fill=True,
        fill_color="#A52A4E",
        fill_opacity=0.60,

        tooltip=(
            f"{row['iom_name']} | "
            f"IDP Returnees since Dec 2024: "
            f"{format_count(idp_returnees_dec2024_value)}"
        ),

        popup=folium.Popup(
            popup_html,
            max_width=320
        )

    ).add_to(idp_returnees_dec2024_layer)


# IDP Returneesレイヤーを地図へ追加
idp_returnees_dec2024_layer.add_to(m)

print(
    "IDP Returnees since Dec 2024 layer created:",
    f"{len(idp_returnees_dec2024_geo):,}",
    "locations"
)

IDP Returnees since Dec 2024 layer created: 957 locations


In [41]:
# 2-13
# Prepare total population data for mapping
# Total Populationデータを地図表示用に準備する

total_population_geo = map_data[
    map_data["total_population"].notna()
    & (map_data["total_population"] > 0)
    & map_data.geometry.notna()
].copy()

print(
    "Total Population locations:",
    f"{len(total_population_geo):,}"
)

print(
    "Maximum Total Population:",
    format_count(
        total_population_geo["total_population"].max()
    )
)

Total Population locations: 2,342
Maximum Total Population: 1,188,608


In [42]:
# 2-14
# Add proportional total population layer
# 総人口を比例円で表示する

# Total Populationレイヤーを作成
total_population_layer = folium.FeatureGroup(
    name="Total Population",
    show=False
)


# Total Populationの最大値を取得
max_total_population = (
    total_population_geo["total_population"].max()
)


# 各地点をTotal Population数に応じた大きさで地図へ追加
for _, row in total_population_geo.iterrows():

    total_population_value = row["total_population"]

    popup_html = f"""
    <div style="
        font-family: Arial;
        font-size: 13px;
        min-width: 220px;
    ">
        <b>Settlement</b><br>
        {row['iom_name']}
        <br><br>

        <b>Residents:</b>
        {format_count(row['residents'])}<br>

        <b>IDPs:</b>
        {format_count(row['idps'])}<br>

        <b>IDP Returnees since Dec 2024:</b>
        {format_count(row['idp_returnees_dec2024'])}<br>

        <b>IDP Returnees in 2026:</b>
        {format_count(row['idp_returnees_2026'])}<br>

        <b>Total Population:</b>
        {format_count(row['total_population'])}
        <br><br>

        <b>Match status:</b>
        {row['match_status']}<br>

        <b>Similarity score:</b>
        {row['similarity_score']}
    </div>
    """

    folium.CircleMarker(
        location=[
            row.geometry.y,
            row.geometry.x
        ],

        radius=proportional_radius(
            total_population_value,
            max_total_population
        ),

        color="#166534",
        weight=1,

        fill=True,
        fill_color="#22c55e",
        fill_opacity=0.55,

        tooltip=(
            f"{row['iom_name']} | "
            f"Total Population: "
            f"{format_count(total_population_value)}"
        ),

        popup=folium.Popup(
            popup_html,
            max_width=320
        )

    ).add_to(total_population_layer)


# Total Populationレイヤーを地図へ追加
total_population_layer.add_to(m)

print(
    "Total Population layer created:",
    f"{len(total_population_geo):,}",
    "locations"
)

Total Population layer created: 2,342 locations


In [43]:
# 2-15
# Prepare rivers data for mapping
# 河川データを読み込み、地図表示用に準備する

rivers = gpd.read_file(
    "syr_rivers_3857.geojson"
)

if rivers.crs is None:
    raise ValueError(
        "River dataset has no CRS information."
    )

print("Original CRS:", rivers.crs)
print("River features:", f"{len(rivers):,}")
print(
    "Missing river geometries:",
    rivers.geometry.isna().sum()
)

print("River geometry types:")
print(
    rivers.geometry.geom_type.value_counts(
        dropna=False
    )
)

Original CRS: EPSG:3857
River features: 12,034
Missing river geometries: 0
River geometry types:
LineString         12023
MultiLineString       11
Name: count, dtype: int64


In [44]:
# 2-16
# Reproject rivers to EPSG:4326
# 河川データをFolium用のEPSG:4326へ変換する

rivers = rivers.to_crs(
    epsg=4326
)

if rivers.crs.to_epsg() != 4326:
    raise ValueError(
        "River data was not converted to EPSG:4326."
    )

print("Converted CRS:", rivers.crs)

Converted CRS: EPSG:4326


In [45]:
# 2-17
# Add rivers layer
# 河川を青いラインとして表示する

# Tooltipに使用できる列を準備
river_tooltip_fields = []
river_tooltip_aliases = []

if "name" in rivers.columns:
    river_tooltip_fields.append("name")
    river_tooltip_aliases.append("River name:")

if "name_en" in rivers.columns:
    river_tooltip_fields.append("name_en")
    river_tooltip_aliases.append(
        "River name (English):"
    )

river_tooltip = (
    folium.GeoJsonTooltip(
        fields=river_tooltip_fields,
        aliases=river_tooltip_aliases,
        sticky=True
    )
    if river_tooltip_fields
    else None
)


# Riversレイヤーを作成
rivers_layer = folium.FeatureGroup(
    name="Rivers",
    show=False
)

folium.GeoJson(
    rivers,
    style_function=lambda feature: {
        "color": "#2F80ED",
        "weight": 1.8,
        "opacity": 0.75
    },
    tooltip=river_tooltip
).add_to(rivers_layer)

rivers_layer.add_to(m)

print(
    "Rivers layer created:",
    f"{len(rivers):,}",
    "features"
)

Rivers layer created: 12,034 features


In [46]:
# 2-18
# Add lakes layer
# 湖沼を青色の面として表示する

lakes = gpd.read_file(
    "syr_lakes_4326.geojson"
)

if lakes.crs is None:
    raise ValueError(
        "Lake dataset has no CRS information."
    )

if lakes.crs.to_epsg() != 4326:
    lakes = lakes.to_crs(epsg=4326)

lakes_layer = folium.FeatureGroup(
    name="Lakes",
    show=False
)

folium.GeoJson(
    lakes,
    style_function=lambda feature: {
        "color": "#2F80ED",
        "weight": 1,
        "fillColor": "#76B7EB",
        "fillOpacity": 0.45
    }
).add_to(lakes_layer)

lakes_layer.add_to(m)

print(
    "Lakes layer created:",
    f"{len(lakes):,}",
    "features"
)

Lakes layer created: 2,925 features


In [47]:
# 2-19
# Add layer control
# 地図上のレイヤー切り替え機能を追加する

folium.LayerControl(
    collapsed=False
).add_to(m);

In [48]:
# 2-20
# Export interactive map
# インタラクティブ地図をHTMLとして保存する

output_map_path = (
    "10_syria_displacement_and_return_atlas.html"
)

m.save(output_map_path)

print(
    "Map saved:",
    output_map_path
)

Map saved: 10_syria_displacement_and_return_atlas.html
